# Week 1 Day 2 assignment: the cheapest consistent agent, and what its mistakes cost

Burak Gülbol

**Challenge:** with a goal of at least 15 / 25 correct *consistently*, using the agent, what is the cheapest model you can find?

**Answer:** the lab's own `openai/gpt-6-luna` at `low`. It scored **17, 16 and 18** on three runs of the queue (worst 16) for about **$0.0074 a run**, and **20 / 25** on the held-out set. The one cheaper-looking candidate I tested, `inclusionai/ling-3.0-flash`, scored 15, 16 and **14**, so it misses the goal, and on the full queue it actually cost **more** than Luna (about $0.0092 a run).

**Beyond the challenge:** at under a cent a run, the model bill is not where the money is. Over three runs, Luna's agent approved **about $814 a run** of claims that a handler would have referred for review, overpaid about **$288 a run**, and underpaid about **$112 a run**. Ling, two points less accurate on average, approved about **$1,229 a run** without review. The size and kind of mistakes matter far more than a fraction of a cent of tokens.

| Model (Ed's agent) | Right, 3 runs | Worst | Cost / run | Paid without review / run | Overpaid / run | Underpaid / run |
|---|---|---|---|---|---|---|
| **gpt-6-luna (low)** | 17, 16, 18 | **16** | **$0.0074** | $814 | $288 | $112 |
| ling-3.0-flash | 15, 16, 14 | 14 | $0.0092 | $1,229 | $239 | $60 |
| gpt-6-luna (low), held-out, 1 run | 20 | 20 | $0.0092 | $420 | $0 | $30 |

The dollar figures come from a handful of claims over three runs, so read them as direction, not precision. "Paid without review" is exposure, not loss: some of those claims might have been approved after a handler's review.

## Approach

1. Use the lab's agent exactly as it is: same prompt, same `get_policy` and `record_decision` tools, same scoring (`is_right`: decision matches and payout within a dollar).
2. Run every configuration **three times** on the 25-claim queue and judge it on its **worst** run, because both accuracy and cost vary from run to run.
3. Measure cost from what OpenRouter actually billed, never from the price list.
4. Pick candidates using the other contributions in this folder. They had already screened most models cheaper than Luna (`gpt-oss-20b`, `qwen3.7-flash`, `ling-3.0-flash` and others); only `ling-3.0-flash` was close enough to the goal to be worth three more runs.
5. Go beyond the score: break the mistakes down by kind and by dollar amount.
6. Finish with one run of the winner on the held-out set, which nothing above was tuned on.

New OpenRouter accounts are limited to 20 requests a minute per model, so the loops pause between claims.

## Setup

In [2]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


In [5]:
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [6]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


## The lab's agent, unchanged

The instructions, the two tools and the agent are copied from `labs/week1day2.ipynb`.

In [14]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

In [17]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

In [25]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [26]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

In [29]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }

## Baseline: Luna at `low`, three runs

Each claim takes three model calls (ask for the policy, record the decision, finish), so the loop waits 9 seconds after each claim to stay under the rate limit.

`recorded` is cleared before each run. Without that, a claim the agent forgot to record in run 2 would silently be scored with run 1's decision.

In [30]:
agent_runs = []

for i in range(3):
    recorded.clear()
    results = []

    for claim in claims:
        results.append(decide_with_agent(claim))    # run the agent on one claim, score it, keep it
        time.sleep(9)

    right = sum(r["right"] for r in results)
    cost = sum(r["cost"] for r in results)

    print(f"Run {i + 1}: {right} of {len(results)} right for ${cost:.4f}")
    agent_runs.append(results)

Run 1: 17 of 25 right for $0.0089
Run 2: 16 of 25 right for $0.0067
Run 3: 18 of 25 right for $0.0064


## What kind of mistakes?

Rows are the handler's decision, columns the agent's, over all 75 decisions (3 runs x 25 claims).

- No wrong declines and no approvals of claims the handler declined.
- 7 needless referrals (handler approved or declined, agent referred): these cost a handler's time.
- **10 approvals of claims the handler referred**: money paid on claims that needed a closer look.

In [31]:
agent_results = agent_runs[0]     # later lab cells expect this name

all_results = [r for run in agent_runs for r in run]     # flatten the 3 runs into one list
df = pd.DataFrame(all_results)
df["handler"] = df["claim_id"].map(lambda c: labels[c]["decision"])

pd.crosstab(df["handler"], df["decision"], margins=True)

decision,approve,decline,refer,All
handler,,,,
approve,36,0,6,42
decline,0,14,1,15
refer,10,0,8,18
All,46,14,15,75


### Which claims, and how much money?

The mistakes are **systematic, not random**: the same claims go wrong on every run.

- The three `manual` claims (CLM-1018, CLM-1019, CLM-1024) were approved on **all three runs**. The right answer depends on the customer's claim history or the claims manual, which this agent cannot see. A bigger model will not fix missing information; a tool that provides it might.
- CLM-1004 and CLM-1021 got the right decision but the **same wrong payout every time** (one underpaid, one overpaid). That is arithmetic and rule application, which code does more reliably than a model.

In [32]:
df["handler_payout"] = df["claim_id"].map(lambda c: labels[c]["payout"])

# 1) Claims the handler referred but the agent approved: which ones, and how much was paid?
risky = df[(df["handler"] == "refer") & (df["decision"] == "approve")]
print("Approved when the handler referred:")
print(risky.groupby(["claim_id", "difficulty"])["payout"].agg(["count", "mean"]))
print(f"Total paid without review: ${risky['payout'].sum():,.2f}\n")

# 2) Right decision, wrong amount
wrong_amount = df[(df["handler"] == df["decision"]) & (~df["right"])]
print("Right decision, wrong amount:")
print(wrong_amount[["claim_id", "payout", "handler_payout"]])

Approved when the handler referred:
                     count   mean
claim_id difficulty              
CLM-1018 manual          3  330.0
CLM-1019 manual          3  210.0
CLM-1024 manual          3  250.0
CLM-1025 hard            1   71.0
Total paid without review: $2,441.00

Right decision, wrong amount:
    claim_id  payout  handler_payout
3   CLM-1004    74.0           186.0
10  CLM-1011  1330.0          1150.0
20  CLM-1021  1053.0           825.0
28  CLM-1004    74.0           186.0
45  CLM-1021  1053.0           825.0
53  CLM-1004    74.0           186.0
70  CLM-1021  1053.0           825.0


## Trying a cheaper model

To swap models, the agent is built by a function instead of being fixed to Luna. Three additions to the lab's version:

- **`try / except`** so one malformed tool call doesn't end a 30-minute run, while still **counting** errors so they can't hide.
- **`cost_missing`**: if a provider doesn't report a cost, treating it as 0 would make that model look free and win the "cheapest" race. Missing costs are counted instead.
- **`calls`**: the number of model calls per claim, which explains cost differences between models.

In [33]:
def make_agent(model, effort=None):
    """Build the lab's agent with the given model and reasoning effort."""
    llm = ChatOpenAI(
        model=model,
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        reasoning_effort=effort,
        timeout=120,
        max_retries=5,
        extra_body={"provider": {"require_parameters": True}},
    )
    return create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)


def decide_with(agent, model, effort, claim):
    """Run one claim through the given agent and score the decision it recorded."""
    start = time.time()
    label = labels[claim["claim_id"]]
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
        messages = result["messages"]
        error = ""
    except Exception as e:
        messages = []
        error = str(e)[:200]
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    costs = [m.response_metadata.get("token_usage", {}).get("cost") for m in messages if m.type == "ai"]
    return {
        "model": model,
        "effort": effort,
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": sum(c or 0 for c in costs),
        "cost_missing": any(c is None for c in costs),
        "calls": len(costs),
        "error": error,
        "seconds": round(time.time() - start, 1),
    }


def run_agent(model, effort=None, runs=3, pause=9):
    """Build the agent, run it over the queue `runs` times and print a summary of each run."""
    agent = make_agent(model, effort)
    all_runs = []
    for i in range(runs):
        recorded.clear()
        results = []
        for claim in claims:
            results.append(decide_with(agent, model, effort, claim))
            time.sleep(pause)
        right = sum(r["right"] for r in results)
        cost = sum(r["cost"] for r in results)
        calls = sum(r["calls"] for r in results)
        errors = sum(1 for r in results if r["error"])
        missing = sum(1 for r in results if r["cost_missing"])
        print(f"{model} ({effort}) run {i + 1}: {right}/25 right, ${cost:.4f}, {calls} calls, {errors} errors, {missing} missing costs")
        all_runs.append(results)
    return all_runs

A one-claim smoke test before spending 30 minutes: right decision, three calls, no error, cost reported.

In [34]:
recorded.clear()
test_agent = make_agent("inclusionai/ling-3.0-flash")
decide_with(test_agent, "inclusionai/ling-3.0-flash", None, claims[0])

{'model': 'inclusionai/ling-3.0-flash',
 'effort': None,
 'technique': 'Agent',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The trip delay claim is covered under Section 3. The flight was canceled due to a crew problem, causing a delay of approximately 19 hours (from scheduled departure June 18 at 11:40 to actual departure June 19 at 06:55), which exceeds the Premium plan's 6-hour waiting period. The claimed expenses ($245 hotel, $95 meals, $80 taxis) are reasonable costs for accommodation, meals, and local transport during the delay. All required documents (carrier cancellation notice and receipts) are attached. The total of $420 is within the Premium plan's $800 maximum for Trip Delay. No deductible applies to Trip Delay per Section 1.",
 'difficulty': 'routine',
 'right': True,
 'cost': 0.0002291562,
 'cost_missing': False,
 'calls': 3,
 'error': '',
 'seconds': 17.2}

### `ling-3.0-flash`, three runs

On the single test claim it looked about 20% cheaper than Luna. It is slower (about 17 seconds a claim on its own), so a 3-second pause is enough to stay under the rate limit.

In [35]:
ling_runs = run_agent("inclusionai/ling-3.0-flash", runs=3, pause=3)

inclusionai/ling-3.0-flash (None) run 1: 15/25 right, $0.0097, 78 calls, 0 errors, 0 missing costs
inclusionai/ling-3.0-flash (None) run 2: 16/25 right, $0.0083, 77 calls, 0 errors, 0 missing costs
inclusionai/ling-3.0-flash (None) run 3: 14/25 right, $0.0096, 75 calls, 0 errors, 0 missing costs


Ling misses the goal on run 3, and over the whole queue it costs **more** than Luna: it made 75 to 78 calls a run instead of 75, and longer replies. One claim is not a cost estimate.

### Side by side, including the cost of mistakes

`exposure` summarizes a set of runs. Its Luna row reproduces the hand-calculated figures above ($814, $288, $112), which checks the refactor.

In [36]:
def exposure(runs, name):
    """Summarize a set of runs: accuracy, model cost and the cost of the mistakes, per run."""
    n = len(runs)
    df = pd.DataFrame([r for run in runs for r in run])
    df["handler"] = df["claim_id"].map(lambda c: labels[c]["decision"])
    df["handler_payout"] = df["claim_id"].map(lambda c: labels[c]["payout"])

    risky = df[(df["handler"] != "approve") & (df["decision"] == "approve")]
    both_approved = df[(df["handler"] == "approve") & (df["decision"] == "approve")]
    diff = both_approved["payout"] - both_approved["handler_payout"]
    extra_refer = df[(df["handler"] != "refer") & (df["decision"] == "refer")]

    return {
        "model": name,
        "worst right": min(sum(r["right"] for r in run) for run in runs),
        "mean right": round(df["right"].sum() / n, 1),
        "cost / run": round(df["cost"].sum() / n, 4),
        "paid without review $ / run": round(risky["payout"].sum() / n),
        "overpaid $ / run": round(diff[diff > 1].sum() / n),
        "underpaid $ / run": round(-diff[diff < -1].sum() / n),
        "needless refers / run": round(len(extra_refer) / n, 1),
    }


pd.DataFrame([exposure(agent_runs, "gpt-6-luna (low)"), exposure(ling_runs, "ling-3.0-flash")])

,model,worst right,mean right,cost / run,paid without review $ / run,overpaid $ / run,underpaid $ / run,needless refers / run
0,gpt-6-luna (low),16,17.0,0.0074,814,288,112,2.3
1,ling-3.0-flash,14,15.0,0.0092,1229,239,60,2.7


Luna is better on every axis that matters here: more accurate, cheaper and faster. Ling makes fewer payout errors but approves far more claims that needed review, which is the more expensive kind of mistake for an insurer.

## Held-out check

One run of the winner on the 25 held-out claims. The functions read `claims` and `labels` as globals, so they are switched to the held-out set and **switched back** straight after.

In [37]:
claims, labels = get_sample_data(held_out=True)       # switch to the held-out set
held_out_runs = run_agent("openai/gpt-6-luna", "low", runs=1, pause=9)
held_out_summary = exposure(held_out_runs, "gpt-6-luna (low), held-out")
claims, labels = get_sample_data()                     # switch BACK to the queue
held_out_summary

openai/gpt-6-luna (low) run 1: 20/25 right, $0.0092, 76 calls, 0 errors, 0 missing costs


{'model': 'gpt-6-luna (low), held-out',
 'worst right': 20,
 'mean right': np.float64(20.0),
 'cost / run': np.float64(0.0092),
 'paid without review $ / run': 420,
 'overpaid $ / run': 0,
 'underpaid $ / run': 30,
 'needless refers / run': 2.0}

The held-out set has the same mix as the queue (15 routine, 5 hard, 5 manual), so the higher score is not an easier set. The `manual` claims are still out of reach: 1 of 5.

In [38]:
pd.DataFrame(held_out_runs[0]).groupby("difficulty")["right"].agg(["sum", "count"])

,sum,count
difficulty,,
hard,4,5
manual,1,5
routine,15,15


## Conclusions

1. **Cheapest consistent model with the lab's agent: `gpt-6-luna` at `low`**, worst run 16 / 25, about $0.0074 a run, 20 / 25 on the held-out set. This matches what others in this folder found, so the result reproduces.
2. **List prices and single tests mislead.** Ling looked cheaper on one claim and was 25% more expensive over the queue.
3. **The score hides the kind of mistake.** Two points of accuracy separated Luna and Ling, but about $415 a run of claims approved without review.
4. **Most remaining errors are systematic.** They come from information the agent doesn't have (customer history, the claims manual) and from arithmetic. A bigger model is the wrong fix for both.

## What I would try next

- A `get_claim_history` tool and a claims-manual lookup for the `manual` claims.
- Work out the payout in code and let the model only classify the items.
- Put the policy record in the prompt and stop after `record_decision`, to drop the two calls that need no judgment.
- Then re-check the cheaper models: with a better agent, the cheapest consistent model may change.